# LeetCode 1805 — Number of Different Integers in a String bằng Python

Link đề: https://leetcode.com/problems/number-of-different-integers-in-a-string/

Độ khó: **Easy** · Chủ đề: Hash Table, String

```python
class Solution:
    def numDifferentIntegers(self, word: str) -> int:

```

**Mục tiêu học:**

1. Biết **tách các khối chữ số liên tiếp** (token) khỏi một chuỗi lẫn chữ và số.
2. Hiểu vấn đề **số 0 ở đầu (leading zeros)**: `"1"`, `"01"`, `"001"` là cùng một số.
3. Dùng `set` để đếm số giá trị **khác nhau**.
4. So sánh các cách: thay chữ bằng khoảng trắng + `split`, quét hai con trỏ, `re.findall`.
5. Hiểu vì sao nên so sánh **chuỗi đã bỏ số 0 đầu** thay vì `int(...)` khi số rất dài (ở ngôn ngữ khác sẽ tràn số).

## 1. Bài toán yêu cầu gì?

Cho chuỗi `word` gồm chữ số và chữ cái thường. Thay **mọi ký tự không phải chữ số** bằng khoảng trắng. Khi đó còn lại các số nguyên ngăn cách bởi ít nhất một khoảng trắng.

Trả về **số lượng số nguyên khác nhau**. Hai số khác nhau nếu biểu diễn thập phân **không có số 0 ở đầu** của chúng khác nhau.

| `word` | Sau khi thay | Các số | Khác nhau | Kết quả |
|---|---|---|---|:---:|
| `"a123bc34d8ef34"` | `" 123  34 8  34"` | 123, 34, 8, 34 | 123, 34, 8 | `3` |
| `"leet1234code234"` | `"    1234    234"` | 1234, 234 | 1234, 234 | `2` |
| `"a1b01c001"` | `" 1 01 001"` | 1, 01, 001 | 1 | `1` |
| `"abc"` | `"   "` | – | – | `0` |
| `"0a00"` | `"0 00"` | 0, 00 | 0 | `1` |

**Ràng buộc:** `1 <= len(word) <= 1000`, chỉ gồm chữ số và chữ thường.

Chú ý: một số có thể dài tới **1000 chữ số**!

In [1]:
examples = [
    ("a123bc34d8ef34", 3),
    ("leet1234code234", 2),
    ("a1b01c001", 1),
]
for word, expected in examples:
    print(f"{word!r:20} -> kỳ vọng {expected}")

'a123bc34d8ef34'     -> kỳ vọng 3
'leet1234code234'    -> kỳ vọng 2
'a1b01c001'          -> kỳ vọng 1


## 2. Nền tảng Python cần biết

### 2.1. `isdigit()` và xây chuỗi mới bằng `join`

Ta mô phỏng đúng mô tả đề: mỗi ký tự không phải số → `' '`. Cách gọn: tạo list ký tự mới rồi `"".join(...)`.

In [2]:
word = "a123bc34d8ef34"
replaced = "".join(ch if ch.isdigit() else " " for ch in word)
print(repr(replaced))

' 123  34 8  34'


`ch if ch.isdigit() else " "` là **biểu thức điều kiện**: giữ `ch` nếu là số, ngược lại thay bằng khoảng trắng.

### 2.2. `str.split()` không đối số

`split()` **không truyền đối số** sẽ cắt theo **mọi chuỗi khoảng trắng liên tiếp** và **bỏ** các mảnh rỗng. Khác với `split(" ")` (cắt tại từng dấu cách, giữ mảnh rỗng).

In [3]:
s = " 123  34 8  34"
print("split()   ->", s.split())
print('split(" ") ->', s.split(" "))

split()   -> ['123', '34', '8', '34']
split(" ") -> ['', '123', '', '34', '8', '', '34']


### 2.3. Leading zeros: `int(...)` và `lstrip("0")`

- `int("001")` → `1`: số 0 đầu tự mất.
- `"001".lstrip("0")` → `"1"`: xóa các ký tự `'0'` ở **bên trái**.
- Cẩn thận: `"000".lstrip("0")` → `""` (chuỗi rỗng) — số `0` biến mất! Ta phải đổi `""` thành `"0"` (hoặc giữ nguyên quy ước `""` đại diện cho số 0, miễn nhất quán).

In [4]:
for t in ["1", "01", "001", "0", "000", "100"]:
    print(f"{t!r:6} int -> {int(t):4} | lstrip -> {t.lstrip('0')!r:6} | lstrip or '0' -> {(t.lstrip('0') or '0')!r}")

'1'    int ->    1 | lstrip -> '1'    | lstrip or '0' -> '1'
'01'   int ->    1 | lstrip -> '1'    | lstrip or '0' -> '1'
'001'  int ->    1 | lstrip -> '1'    | lstrip or '0' -> '1'
'0'    int ->    0 | lstrip -> ''     | lstrip or '0' -> '0'
'000'  int ->    0 | lstrip -> ''     | lstrip or '0' -> '0'
'100'  int ->  100 | lstrip -> '100'  | lstrip or '0' -> '100'


`x or y` trả `x` nếu `x` "truthy", ngược lại trả `y`. Vì `""` là falsy nên `"".lstrip("0") or "0"` → `"0"`. Mẹo này rất hay dùng.

### 2.4. `set` để đếm số giá trị khác nhau

In [5]:
values = [123, 34, 8, 34]
print(set(values), "-> số giá trị khác nhau:", len(set(values)))

{8, 34, 123} -> số giá trị khác nhau: 3


### 2.5. Số nguyên rất lớn trong Python

Python hỗ trợ số nguyên **không giới hạn độ dài**, nên `int("9" * 1000)` vẫn chạy. Nhưng ở C++/Java, số 1000 chữ số **tràn** kiểu `long`. Vì thế lời giải "chuẩn" (dễ chuyển sang ngôn ngữ khác) so sánh **chuỗi đã bỏ 0 đầu**.

In [6]:
big = int("9" * 1000)
print("số chữ số:", len(str(big)))
print("2**63 - 1 (giới hạn long của Java) có", len(str(2**63 - 1)), "chữ số")

số chữ số: 1000
2**63 - 1 (giới hạn long của Java) có 19 chữ số


## 3. Cách trực giác — làm đúng như đề mô tả

1. Thay ký tự không phải số bằng `' '`.
2. `split()` lấy các token số.
3. Đổi mỗi token sang `int` (tự bỏ 0 đầu), cho vào `set`.
4. Trả `len(set)`.

In [7]:
def num_different_integers_split(word: str) -> int:
    replaced = "".join(ch if ch.isdigit() else " " for ch in word)
    tokens = replaced.split()
    return len({int(t) for t in tokens})

for word, expected in examples:
    print(num_different_integers_split(word), "| kỳ vọng", expected)

3 | kỳ vọng 3
2 | kỳ vọng 2
1 | kỳ vọng 1


### 3.1. Điểm yếu của baseline

- Đúng và `O(n)` trong Python — đây là lời giải hoàn toàn chấp nhận được.
- Nhưng `int(t)` với token dài hàng trăm chữ số tốn thêm thời gian chuyển đổi, và **không mang sang được** ngôn ngữ có số nguyên giới hạn.
- Tạo chuỗi `replaced` và list `tokens` → bộ nhớ phụ `O(n)` (không tránh hoàn toàn được vì set cũng cần chứa token).

Câu hỏi: *có thể tự quét để lấy token và chuẩn hóa leading zeros bằng **chuỗi**, không cần `int`?*

## 4. Nút thắt và insight

Hai số (viết không có 0 đầu) bằng nhau **khi và chỉ khi** hai chuỗi chữ số của chúng giống hệt nhau. Vậy:

> **Insight:** Chuẩn hóa mỗi token bằng cách **bỏ 0 đầu** (`lstrip('0')`), rồi đưa **chuỗi** vào `set`. Không cần đổi sang số.

Và để lấy token mà không tạo chuỗi `replaced`, dùng **hai con trỏ**: `i` chạy tới chữ số đầu tiên, `j` chạy tiếp tới hết khối số; token là `word[i:j]`.

In [8]:
# Thử chuẩn hóa bằng chuỗi cho ví dụ 3
tokens = ["1", "01", "001"]
normalized = {t.lstrip("0") or "0" for t in tokens}
print(normalized, "->", len(normalized))

{'1'} -> 1


## 5. Lời giải chính — quét hai con trỏ + chuẩn hóa chuỗi

**Bước 1.** `i = 0`, `seen = set()`.

**Bước 2.** Trong khi `i < n`:
- Nếu `word[i]` không phải số → `i += 1`, tiếp tục.
- Nếu là số: đặt `j = i`, tăng `j` khi `word[j]` còn là số. Token là `word[i:j]`.
- Chuẩn hóa: `key = token.lstrip("0") or "0"`, thêm vào `seen`.
- Nhảy `i = j` (bỏ qua cả khối vừa đọc).

**Bước 3.** Trả `len(seen)`.

In [9]:
# Đoạn 1: tìm một khối số bắt đầu tại i
word = "a123bc34"
i = 1
j = i
while j < len(word) and word[j].isdigit():
    j += 1
print("i =", i, "j =", j, "token =", word[i:j])

i = 1 j = 4 token = 123


In [10]:
# Ghép thành hàm hoàn chỉnh
def num_different_integers(word: str) -> int:
    n = len(word)
    seen = set()
    i = 0
    while i < n:
        if not word[i].isdigit():
            i += 1
            continue
        j = i
        while j < n and word[j].isdigit():
            j += 1
        token = word[i:j]
        seen.add(token.lstrip("0") or "0")
        i = j
    return len(seen)

for word, expected in examples:
    print(num_different_integers(word), "| kỳ vọng", expected)

3 | kỳ vọng 3
2 | kỳ vọng 2
1 | kỳ vọng 1


Chú ý `j < n and word[j].isdigit()`: phải kiểm tra `j < n` **trước** — Python đánh giá `and` từ trái sang phải và dừng sớm (short-circuit), nên không bao giờ đọc `word[n]` (vượt chỉ số).

## 6. TRACE từng bước

Trace `word = "a1b01c001"`:

| i (đầu khối) | j (sau khối) | token | key (bỏ 0 đầu) | seen |
|---|---|---|---|---|
| 1 | 2 | `"1"` | `"1"` | {"1"} |
| 3 | 5 | `"01"` | `"1"` | {"1"} |
| 6 | 9 | `"001"` | `"1"` | {"1"} |

Kết quả `1`.

In [11]:
def trace_num_different_integers(word: str) -> int:
    print(f"=== word = {word!r} ===")
    n = len(word)
    seen = set()
    i = 0
    while i < n:
        if not word[i].isdigit():
            i += 1
            continue
        j = i
        while j < n and word[j].isdigit():
            j += 1
        token = word[i:j]
        key = token.lstrip("0") or "0"
        is_new = key not in seen
        seen.add(key)
        print(f"khối [{i}:{j}] token={token!r:7} key={key!r:6} mới? {is_new!s:5} seen={sorted(seen)}")
        i = j
    print("-> số số khác nhau:", len(seen))
    return len(seen)

trace_num_different_integers("a1b01c001")
print()
trace_num_different_integers("a123bc34d8ef34")

=== word = 'a1b01c001' ===
khối [1:2] token='1'     key='1'    mới? True  seen=['1']
khối [3:5] token='01'    key='1'    mới? False seen=['1']
khối [6:9] token='001'   key='1'    mới? False seen=['1']
-> số số khác nhau: 1

=== word = 'a123bc34d8ef34' ===
khối [1:4] token='123'   key='123'  mới? True  seen=['123']
khối [6:8] token='34'    key='34'   mới? True  seen=['123', '34']
khối [9:10] token='8'     key='8'    mới? True  seen=['123', '34', '8']
khối [12:14] token='34'    key='34'   mới? False seen=['123', '34', '8']
-> số số khác nhau: 3


3

### 6.1. Trace nhánh đặc biệt: số 0 và khối ở cuối chuỗi

In [12]:
trace_num_different_integers("0a00b000c0")
print()
trace_num_different_integers("abc")
print()
trace_num_different_integers("12")

=== word = '0a00b000c0' ===
khối [0:1] token='0'     key='0'    mới? True  seen=['0']
khối [2:4] token='00'    key='0'    mới? False seen=['0']
khối [5:8] token='000'   key='0'    mới? False seen=['0']
khối [9:10] token='0'     key='0'    mới? False seen=['0']
-> số số khác nhau: 1

=== word = 'abc' ===
-> số số khác nhau: 0

=== word = '12' ===
khối [0:2] token='12'    key='12'   mới? True  seen=['12']
-> số số khác nhau: 1


1

Với `"000"`, `lstrip("0")` cho `""` và `or "0"` sửa thành `"0"`. Khối `"12"` ở **cuối chuỗi** được đọc đúng nhờ điều kiện `j < n`.

## 7. Các lời giải khác

### Cách 3 — Biểu thức chính quy `re.findall(r"\d+", word)`

Module `re` (regular expression) có sẵn trong thư viện chuẩn. `\d` khớp một chữ số, `+` nghĩa là "một hoặc nhiều". `findall` trả về list mọi khối số.

In [13]:
import re
print(re.findall(r"\d+", "a123bc34d8ef34"))

def num_different_integers_regex(word: str) -> int:
    return len({t.lstrip("0") or "0" for t in re.findall(r"\d+", word)})

for word, expected in examples:
    print(num_different_integers_regex(word), "| kỳ vọng", expected)

['123', '34', '8', '34']
3 | kỳ vọng 3
2 | kỳ vọng 2
1 | kỳ vọng 1


Chuỗi `r"..."` là **raw string**: dấu `\` được giữ nguyên, rất tiện cho regex.

### Cách 4 — Xây token từng ký tự với biến `cur` + ký tự gác

Duyệt `word + "#"` (thêm ký tự gác không phải số ở cuối, để khối cuối luôn được "chốt").

In [14]:
def num_different_integers_builder(word: str) -> int:
    seen = set()
    cur = ""
    for ch in word + "#":
        if ch.isdigit():
            cur += ch
        elif cur:
            seen.add(cur.lstrip("0") or "0")
            cur = ""
    return len(seen)

for word, expected in examples:
    print(num_different_integers_builder(word), "| kỳ vọng", expected)

3 | kỳ vọng 3
2 | kỳ vọng 2
1 | kỳ vọng 1


### 7.1. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Thay + `split` + `int` | làm đúng mô tả đề | `O(n)`* | `O(n)` | dễ nhất, cần số nguyên lớn |
| **2. Hai con trỏ + `lstrip` (chính)** | token bằng slicing | `O(n)` | `O(n)` | mang sang ngôn ngữ khác được |
| 3. `re.findall` | regex `\d+` | `O(n)` | `O(n)` | ngắn nhất, cần biết regex |
| 4. Builder + ký tự gác | cộng dồn `cur` | `O(n)` trung bình** | `O(n)` | trực quan |

`n = len(word)`. *`int()` của chuỗi rất dài có thể tốn hơn tuyến tính một chút trong CPython, với `n <= 1000` không đáng kể. **`cur += ch` có thể tạo chuỗi mới mỗi lần; CPython thường tối ưu được, trường hợp xấu `O(L²)` với khối dài `L`.

Bộ nhớ `O(n)` là do `set` chứa các token (tổng độ dài ≤ `n`).

## 8. Code nộp LeetCode

In [15]:
class Solution:
    def numDifferentIntegers(self, word: str) -> int:
        n = len(word)
        seen = set()
        i = 0
        while i < n:
            if word[i].isdigit():
                j = i
                while j < n and word[j].isdigit():
                    j += 1
                seen.add(word[i:j].lstrip("0") or "0")
                i = j
            else:
                i += 1
        return len(seen)

sol = Solution()
for word, expected in examples:
    print(sol.numDifferentIntegers(word), "| kỳ vọng", expected)

3 | kỳ vọng 3
2 | kỳ vọng 2
1 | kỳ vọng 1


## 9. Vì sao thuật toán đúng?

1. **Tách đúng các số:** mỗi lần vào nhánh số, `i` là chữ số **đầu** của một khối (vì ký tự trước `i` hoặc không tồn tại, hoặc không phải số — do ta chỉ dừng `j` ở ký tự không phải số / hết chuỗi, rồi gán `i = j`). Vòng `while j` kéo tới hết khối. Vậy `word[i:j]` đúng bằng một số trong chuỗi sau khi thay ký tự → khoảng trắng, và mỗi khối được đọc đúng một lần.
2. **So sánh đúng:** hai chuỗi chữ số biểu diễn cùng một số nguyên ⇔ sau khi bỏ hết `0` đầu chúng giống nhau (riêng số 0 quy về `"0"`). Đây chính là định nghĩa "khác nhau" của đề.
3. `set` giữ mỗi khóa một lần → `len(seen)` là số số khác nhau.

**Invariant:** trước mỗi vòng lặp ngoài, `seen` chứa đúng khóa chuẩn hóa của mọi khối số nằm hoàn toàn trong `word[0:i]`.

## 10. Độ phức tạp

`n = len(word)`.

- **Time:** `O(n)` — `i` và `j` chỉ tiến về phía trước, mỗi ký tự được xét `O(1)` lần; slicing và `lstrip` tốn tỉ lệ với độ dài khối, tổng các khối ≤ `n`. Thêm vào `set` băm chuỗi độ dài `L` tốn `O(L)` (trung bình).
- **Space:** `O(n)` cho `set` (trường hợp xấu: nhiều số khác nhau).

## 11. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.numDifferentIntegers("a123bc34d8ef34"), 3)
check("đề 2", sol.numDifferentIntegers("leet1234code234"), 2)
check("đề 3", sol.numDifferentIntegers("a1b01c001"), 1)
check("không có số", sol.numDifferentIntegers("abc"), 0)
check("chỉ số 0 nhiều dạng", sol.numDifferentIntegers("0a00b000"), 1)
check("0 và 10 khác nhau", sol.numDifferentIntegers("0a10"), 2)
check("1 ký tự số", sol.numDifferentIntegers("7"), 1)
check("số ở đầu và cuối", sol.numDifferentIntegers("12abc12"), 1)
check("số rất dài", sol.numDifferentIntegers("0" * 500 + "1" + "x" + "1"), 1)
check("10 vs 100", sol.numDifferentIntegers("10a100a010"), 2)

PASS đề 1: 3
PASS đề 2: 2
PASS đề 3: 1
PASS không có số: 0
PASS chỉ số 0 nhiều dạng: 1
PASS 0 và 10 khác nhau: 2
PASS 1 ký tự số: 1
PASS số ở đầu và cuối: 1
PASS số rất dài: 1
PASS 10 vs 100: 2


In [17]:
import random, string
random.seed(1805)
solvers = [num_different_integers_split, num_different_integers, num_different_integers_regex,
           num_different_integers_builder, Solution().numDifferentIntegers]
for _ in range(3000):
    n = random.randint(1, 25)
    word = "".join(random.choice("0001ab" if random.random() < 0.5 else string.ascii_lowercase + string.digits)
                   for _ in range(n))
    answers = [f(word) for f in solvers]
    assert len(set(answers)) == 1, (word, answers)
print("5 cách khớp nhau trên 3000 chuỗi ngẫu nhiên (seed 1805, nhiều số 0)")

5 cách khớp nhau trên 3000 chuỗi ngẫu nhiên (seed 1805, nhiều số 0)


Bộ sinh ngẫu nhiên cố ý chọn bảng chữ `"0001ab"` một nửa số lần để tạo **nhiều số 0** — nơi dễ sai nhất.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Không bỏ số 0 đầu khi so sánh chuỗi

In [18]:
def wrong_no_strip(word):
    return len(set(re.findall(r"\d+", word)))

print("sai:", wrong_no_strip("a1b01c001"), "| đúng:", num_different_integers("a1b01c001"))

sai: 3 | đúng: 1


### Lỗi 2: `lstrip("0")` làm mất số 0

Nếu chỉ `lstrip("0")` mà không `or "0"`, các số `0`, `00` thành `""` — vẫn **đếm đúng** vì mọi số 0 cùng thành `""`! Nhưng nếu sau đó in ra/so sánh với số khác thì dễ nhầm. Lỗi thật sự xảy ra khi bỏ qua token rỗng:

In [19]:
def wrong_skip_empty(word):
    seen = set()
    for t in re.findall(r"\d+", word):
        key = t.lstrip("0")
        if key:                 # vô tình bỏ qua số 0!
            seen.add(key)
    return len(seen)

print("sai:", wrong_skip_empty("a0b1"), "| đúng:", num_different_integers("a0b1"))

sai: 1 | đúng: 2


### Lỗi 3: Dùng `split(" ")` thay vì `split()`

`split(" ")` sinh mảnh rỗng `''` → `int('')` báo `ValueError`.

In [20]:
replaced = " 123  34"
try:
    print({int(t) for t in replaced.split(" ")})
except ValueError as e:
    print("ValueError:", e)
print("đúng:", {int(t) for t in replaced.split()})

ValueError: invalid literal for int() with base 10: ''
đúng: {34, 123}


### Lỗi 4: Đếm từng chữ số thay vì từng **số**

`len(set(ch for ch in word if ch.isdigit()))` đếm các **chữ số** khác nhau (`"a123"` → 3), không phải số nguyên (`123` → 1).

### Lỗi 5: Quên chốt khối số ở cuối chuỗi (cách builder)

Không thêm ký tự gác `"#"` và không xử lý `cur` sau vòng lặp → `"ab12"` trả `0`.

In [21]:
def wrong_builder_no_sentinel(word):
    seen, cur = set(), ""
    for ch in word:
        if ch.isdigit():
            cur += ch
        elif cur:
            seen.add(cur.lstrip("0") or "0")
            cur = ""
    return len(seen)

print("sai:", wrong_builder_no_sentinel("ab12"), "| đúng:", num_different_integers("ab12"))

sai: 0 | đúng: 1


## 13. Mẹo nhận dạng pattern

- "Chuỗi lẫn chữ và số, lấy **các số**" → tách **token** bằng hai con trỏ / `split()` / `re.findall(r"\d+")`.
- "**Khác nhau / distinct**" → `set` + **chuẩn hóa khóa** trước khi thêm (bỏ 0 đầu, chữ thường hóa, sắp xếp ký tự…).
- Số có thể rất dài → so sánh dạng **chuỗi chuẩn hóa** thay vì đổi sang số.
- Bài liên quan: 2042 Check if Numbers Are Ascending in a Sentence, 1796 Second Largest Digit in a String, 49 Group Anagrams (chuẩn hóa khóa), 929 Unique Email Addresses (chuẩn hóa rồi đếm distinct).

## 14. Biến thể

### 14.1. Trả về chính danh sách các số khác nhau, sắp xếp tăng dần

So sánh số dạng chuỗi theo **(độ dài, chuỗi)**: số ít chữ số hơn thì nhỏ hơn; cùng độ dài thì so từ điển — không cần `int`.

In [22]:
def sorted_distinct_integers(word: str):
    keys = {t.lstrip("0") or "0" for t in re.findall(r"\d+", word)}
    return sorted(keys, key=lambda k: (len(k), k))

print(sorted_distinct_integers("a123bc34d8ef34x0009"))
print(sorted(["123", "34", "8", "9"]), "<- sắp xếp chuỗi thường sẽ sai thứ tự số")

['8', '9', '34', '123']
['123', '34', '8', '9'] <- sắp xếp chuỗi thường sẽ sai thứ tự số


`key=lambda k: (len(k), k)` — Python so sánh **tuple** theo từng phần tử từ trái sang: trước là độ dài, hòa thì so chuỗi.

### 14.2. Đếm số lần xuất hiện của mỗi số (dùng `Counter`)

In [23]:
from collections import Counter

def count_integers(word: str):
    return Counter(t.lstrip("0") or "0" for t in re.findall(r"\d+", word))

print(count_integers("a123bc34d8ef34"))

Counter({'34': 2, '123': 1, '8': 1})


## 15. Bài tập luyện tập

**Bài 1 (đoán output):** Không chạy code, hãy đoán `re.findall(r"\d+", "x00y0z105")` và `["00".lstrip("0") or "0", "105".lstrip("0")]`.

**Bài 2 (tính tay):** Lập bảng trace (i, j, token, key) cho `word = "ab0012cd12e3"`. Đáp án?

**Bài 3 (điền TODO):** Hoàn thiện hàm tách token bằng hai con trỏ, trả về list token (chưa chuẩn hóa).

In [24]:
def ex3_tokens(word: str) -> list:
    # TODO 1: tokens = [], i = 0, n = len(word)
    # TODO 2: nếu word[i] không là số -> i += 1
    # TODO 3: nếu là số: j chạy tới hết khối, tokens.append(word[i:j]), i = j
    pass

# assert ex3_tokens("a123bc34d8ef34") == ["123", "34", "8", "34"]
# assert ex3_tokens("12ab") == ["12"]

**Bài 4 (tự viết):** Viết `ex4_sum_of_integers(word)` — tổng **tất cả** các số trong chuỗi (kể cả trùng).

In [25]:
def ex4_sum_of_integers(word: str) -> int:
    # TODO
    pass

# assert ex4_sum_of_integers("a123bc34d8ef34") == 199
# assert ex4_sum_of_integers("abc") == 0

**Bài 5 (biến thể):** Viết `ex5_largest_integer(word)` trả về **chuỗi** biểu diễn số lớn nhất (không có 0 đầu) mà **không dùng `int`**; không có số thì trả `""`.

In [26]:
def ex5_largest_integer(word: str) -> str:
    # TODO: so sánh theo (len, chuỗi)
    pass

# assert ex5_largest_integer("a99b100c0099") == "100"
# assert ex5_largest_integer("x007") == "7"
# assert ex5_largest_integer("abc") == ""

**Bài 6 (mini challenge):** Hai chuỗi `w1`, `w2`. Trả về số lượng số nguyên (khác nhau) xuất hiện trong **cả hai** chuỗi.

In [27]:
def ex6_common_integers(w1: str, w2: str) -> int:
    # TODO
    pass

# assert ex6_common_integers("a1b02c3", "x001y3z4") == 2
# assert ex6_common_integers("abc", "1") == 0

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `\d+` là khối số liên tiếp | `lstrip` chỉ xóa bên trái | `['00', '0', '105']`, `['0', '105']` |
| 2 | 3 khối | `"0012"` → `"12"` | đáp án 2 |
| 3 | giống lời giải chính | bỏ phần chuẩn hóa | `tokens.append(word[i:j])` |
| 4 | tách token | `int(t)` cộng dồn | `sum(int(t) for t in ...)` |
| 5 | chuẩn hóa trước | `max(keys, key=lambda k: (len(k), k))` | rỗng → `""` |
| 6 | hai set | phép giao `&` | `len(set1 & set2)` |

## 17. Đáp án tham khảo

**Bài 1:** `['00', '0', '105']` và `['0', '105']`.

**Bài 2:** khối `[2:6]` `"0012"` → `"12"`; khối `[8:10]` `"12"` → `"12"` (trùng); khối `[11:12]` `"3"` → `"3"`. Đáp án **2**.

In [28]:
print(re.findall(r"\d+", "x00y0z105"), ["00".lstrip("0") or "0", "105".lstrip("0")])
assert trace_num_different_integers("ab0012cd12e3") == 2

def ex3_tokens_ans(word):
    tokens, i, n = [], 0, len(word)
    while i < n:
        if not word[i].isdigit():
            i += 1
            continue
        j = i
        while j < n and word[j].isdigit():
            j += 1
        tokens.append(word[i:j])
        i = j
    return tokens

def ex4_sum_of_integers_ans(word):
    return sum(int(t) for t in ex3_tokens_ans(word))

def ex5_largest_integer_ans(word):
    keys = {t.lstrip("0") or "0" for t in ex3_tokens_ans(word)}
    if not keys:
        return ""
    return max(keys, key=lambda k: (len(k), k))

def ex6_common_integers_ans(w1, w2):
    k1 = {t.lstrip("0") or "0" for t in ex3_tokens_ans(w1)}
    k2 = {t.lstrip("0") or "0" for t in ex3_tokens_ans(w2)}
    return len(k1 & k2)

assert ex3_tokens_ans("a123bc34d8ef34") == ["123", "34", "8", "34"]
assert ex3_tokens_ans("12ab") == ["12"]
assert ex4_sum_of_integers_ans("a123bc34d8ef34") == 199
assert ex4_sum_of_integers_ans("abc") == 0
assert ex5_largest_integer_ans("a99b100c0099") == "100"
assert ex5_largest_integer_ans("x007") == "7"
assert ex5_largest_integer_ans("abc") == ""
assert ex6_common_integers_ans("a1b02c3", "x001y3z4") == 2
assert ex6_common_integers_ans("abc", "1") == 0
print("Đáp án tham khảo PASS toàn bộ test!")

['00', '0', '105'] ['0', '105']
=== word = 'ab0012cd12e3' ===
khối [2:6] token='0012'  key='12'   mới? True  seen=['12']
khối [8:10] token='12'    key='12'   mới? False seen=['12']
khối [11:12] token='3'     key='3'    mới? True  seen=['12', '3']
-> số số khác nhau: 2
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def numDifferentIntegers(self, word: str) -> int:
        n, i, seen = len(word), 0, set()
        while i < n:
            if word[i].isdigit():
                j = i
                while j < n and word[j].isdigit():   # kiểm tra j < n TRƯỚC
                    j += 1
                seen.add(word[i:j].lstrip("0") or "0")   # chuẩn hóa leading zeros
                i = j
            else:
                i += 1
        return len(seen)
```

Phiên bản ngắn: `len({t.lstrip("0") or "0" for t in re.findall(r"\d+", word)})`.

**Checklist:**

- [ ] Đã bỏ số 0 đầu (`"01"` ≡ `"1"`)?
- [ ] Số `0`/`00` vẫn được đếm (không bị bỏ vì chuỗi rỗng)?
- [ ] Khối số ở **cuối** chuỗi được đọc?
- [ ] Dùng `split()` không đối số nếu đi theo cách thay khoảng trắng?
- [ ] Đếm **số**, không đếm **chữ số**?